# 🔍 Restaurant Vector Search — Semantic Embedding & Retrieval
### LA Luxury Restaurant Recommendation System — Phase 2

**Purpose:** Convert the cleaned restaurant dataset into a semantic vector database  
using HuggingFace sentence embeddings and ChromaDB, then build a retrieval  
function that returns the best restaurant matches for any natural language query.

**Input:**  `cleaned_restaurants_final.csv`  
**Output:** `./chroma_restaurants/` — persisted vector database ready for the Gradio UI  

---
**Pipeline Overview:**
```
cleaned_restaurants_final.csv
        │
        ▼
  restaurant_metadata  (rich text per restaurant)
        │
        ▼
  Export → tagged_restaurant_descriptions.txt
        │
        ▼
  TextLoader + CharacterTextSplitter  (one doc per restaurant)
        │
        ▼
  HuggingFace Embeddings  (sentence-transformers/all-MiniLM-L6-v2)
        │
        ▼
  ChromaDB  →  persist to ./chroma_restaurants/
        │
        ▼
  retrieve_semantic_recommendations(query)  →  DataFrame of results
```

## 📦 Cell 1 — Import Libraries

In [1]:
import os
import re
import warnings
import pandas as pd

from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma 

# Optional — only needed if switching to OpenAI embeddings for production
# from langchain_openai import OpenAIEmbeddings

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 220)

print("✅ Libraries loaded successfully.")

d:\GitProjects\RestaurantRecommenderLLM\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✅ Libraries loaded successfully.


## 🔑 Cell 2 — Load Environment Variables

Loads your `.env` file for API keys (Anthropic, OpenAI if used, etc.)  
Create a `.env` file in the same folder as this notebook:
```
ANTHROPIC_API_KEY=your_key_here
OPENAI_API_KEY=your_key_here   # only if using OpenAI embeddings
```

In [2]:
load_dotenv()

anthropic_key = os.getenv("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY loaded:", "✅ Yes" if anthropic_key else "⚠️  Not found — check your .env file")

ANTHROPIC_API_KEY loaded: ✅ Yes


## 📂 Cell 3 — Load the Cleaned Restaurant Dataset

> **Note:** Place `cleaned_restaurants_final.csv` in the same folder as this notebook,  
> or update `CSV_PATH` below to point to its location.

In [3]:
# ---------------------------------------------------------
# UPDATE THIS PATH if your CSV is in a different location
# ---------------------------------------------------------
CSV_PATH = "../data/cleaned_restaurants_final.csv"

restaurants = pd.read_csv(CSV_PATH)

print(f"✅ Dataset loaded: {len(restaurants)} restaurants, {len(restaurants.columns)} columns")
print(f"\nColumns: {restaurants.columns.tolist()}")

✅ Dataset loaded: 94 restaurants, 15 columns

Columns: ['Name', 'Location', 'Description', 'Address', 'Telephone Number', 'Price', 'Cuisine Type', 'Dining Atmosphere', 'Sky-High Rooftop', 'Michelin-Guide', 'Customer Ratings', 'Operation Hours', 'Reservations', 'Dress Code', 'restaurant_metadata']


## 👀 Cell 4 — Preview the Dataset

In [4]:
# Preview key columns
restaurants[[
    'Name', 'Location', 'Cuisine Type', 'Dining Atmosphere',
    'Michelin-Guide', 'Price', 'Customer Ratings', 'Sky-High Rooftop'
]].head(10)

,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop
0,Somni,West Hollywood,Spanish Modernist,Fine-Dining,3-Star,$$$$$,5.0,No
1,Providence,Hollywood,Seafood / Contemporary,Fine-Dining,3-Star,$$$$,5.0,No
2,Hayato,Downtown Arts District,Japanese Kaiseki,Fine-Dining,2-Star,$$$$,5.0,No
3,n/naka,Palms,Japanese Kaiseki,Fine-Dining,2-Star,$$$$,5.0,No
4,Melisse,Santa Monica,French / Contemporary American,Fine-Dining,2-Star,$$$$,4.0,No
5,Vespertine,Culver City,Contemporary American / Innovative,Fine-Dining,2-Star,$$$$,5.0,No
6,Sushi Kaneyoshi,Little Tokyo,Japanese / Sushi,Fine-Dining,1-Star,$$$$,5.0,No
7,Restaurant Ki,Little Tokyo,Korean Contemporary,Fine-Dining,1-Star,$$$$,5.0,No
8,715 Sushi,Downtown Los Angeles,Japanese / Sushi,Fine-Dining,1-Star,$$$$,5.0,No
9,Orsa & Winston,Downtown Los Angeles,Contemporary,Fine-Dining,1-Star,$$$$,4.8,No


## 📝 Cell 5 — Inspect the `restaurant_metadata` Column

The `restaurant_metadata` column is the **core text field** that will be embedded.  
It combines all key attributes into a single rich natural-language string,  
allowing the embedding model to capture full semantic context per restaurant.

In [5]:
print("=== restaurant_metadata COLUMN PREVIEW ===")
print(f"Total entries: {restaurants['restaurant_metadata'].notna().sum()}")
print()

# Show a sample across different Michelin tiers and price ranges
sample_indices = [0, 5, 13, 26, 36, 51, 65]  # Somni, Vespertine, Holbox, Maccheroni, 71Above, Badmaash, Belvedere
for idx in sample_indices:
    row = restaurants.iloc[idx]
    print(f"[{idx}] {row['Name']} ({row['Michelin-Guide']}, {row['Price']})")
    print(f"     {row['restaurant_metadata']}")
    print()

=== restaurant_metadata COLUMN PREVIEW ===
Total entries: 94

[0] Somni (3-Star, $$$$$)
     Somni is a Spanish Modernist restaurant located in West Hollywood, Los Angeles. A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating per night. Price range: $$$$$. Atmosphere: Fine-Dining. Michelin Guide: Michelin 3-Star. Customer Rating: 5.0/5.

[5] Vespertine (2-Star, $$$$)
     Vespertine is a Contemporary American / Innovative restaurant located in Culver City, Los Angeles. An avant-garde two-Michelin-star dining experience from Chef Jordan Kahn set inside a wavy obelisk architectural marvel with a sci-fi dreamscape atmosphere. Price range: $$$$. Atmosphere: Fine-Dining. Michelin Guide: Michelin 2-Star. Customer Rating: 5.0/5.

[13] Holbox (1-Star, $$)
     Holbox is a Mexican / Seafood restaurant located in South Los Angeles, Los Angeles. A Yucatecan-style seafood counter from Chef Gilberto Cetina Jr. inside Mercado La Palo

## 💾 Cell 6 — Export Metadata to Text File for LangChain Loading

LangChain's `TextLoader` reads from a plain text file, one entry per line.  
We export the `restaurant_metadata` column — one restaurant per line —  
which `CharacterTextSplitter` will then split into individual documents.

In [6]:
TXT_PATH = "../data/tagged_restaurant_descriptions.txt"

restaurants["restaurant_metadata"].to_csv(
    TXT_PATH,
    index=False,
    header=False
)

# Verify output
with open(TXT_PATH, "r", encoding="utf-8") as f:
    lines = f.readlines()

print(f"✅ Exported {len(lines)} lines to '{TXT_PATH}'")
print(f"\nFirst entry preview:")
print(f"  {lines[0].strip()}")
print(f"\nLast entry preview:")
print(f"  {lines[-1].strip()}")

✅ Exported 94 lines to '../data/tagged_restaurant_descriptions.txt'

First entry preview:
  "Somni is a Spanish Modernist restaurant located in West Hollywood, Los Angeles. A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor Zabala with one seating per night. Price range: $$$$$. Atmosphere: Fine-Dining. Michelin Guide: Michelin 3-Star. Customer Rating: 5.0/5."

Last entry preview:
  "Catch LA is a Japanese /  New American restaurant located in West Hollywood, Los Angeles. Catch LA enters a new era with a renewed food-first mentality and a sharpened focus on service & experience. The restaurant debuts a refreshed design alongside an evolved menu built around its iconic seafood, steak, and sushi program, with fish flown in from Tokyo’s Toyosu Market Price range: $$$. Atmosphere: Romantic / Smart-Casual. Michelin Guide: No. Customer Rating: 4.7/5. Rooftop/top-floor dining."


## 📄 Cell 7 — Load & Split Documents with LangChain

**Strategy:** `chunk_size=1` with `separator="\n"` means each newline-separated  
restaurant entry becomes exactly **one LangChain Document** — preserving each  
restaurant as a single atomic unit for embedding and retrieval.

In [7]:
# Load the raw text file
raw_documents = TextLoader(TXT_PATH, encoding="utf-8").load()

# Split into one document per restaurant (each line = one restaurant)
text_splitter = CharacterTextSplitter(
    chunk_size=1,
    chunk_overlap=0,    # No overlap — each restaurant is self-contained
    separator="\n"
)
documents = text_splitter.split_documents(raw_documents)

print(f"✅ Created {len(documents)} LangChain documents")
print(f"   Expected: {len(restaurants)} (one per restaurant)")
print()
print("=== SAMPLE DOCUMENTS ===")
for i in [0, 1, 2]:
    print(f"\n[Document {i}]")
    print(f"  Content : {documents[i].page_content[:120]}...")
    print(f"  Metadata: {documents[i].metadata}")

Created a chunk of size 314, which is longer than the specified 1
Created a chunk of size 352, which is longer than the specified 1
Created a chunk of size 345, which is longer than the specified 1
Created a chunk of size 358, which is longer than the specified 1
Created a chunk of size 355, which is longer than the specified 1
Created a chunk of size 358, which is longer than the specified 1
Created a chunk of size 320, which is longer than the specified 1
Created a chunk of size 335, which is longer than the specified 1
Created a chunk of size 328, which is longer than the specified 1
Created a chunk of size 311, which is longer than the specified 1
Created a chunk of size 316, which is longer than the specified 1
Created a chunk of size 323, which is longer than the specified 1
Created a chunk of size 337, which is longer than the specified 1
Created a chunk of size 343, which is longer than the specified 1
Created a chunk of size 360, which is longer than the specified 1
Created a 

✅ Created 94 LangChain documents
   Expected: 94 (one per restaurant)

=== SAMPLE DOCUMENTS ===

[Document 0]
  Content : "Somni is a Spanish Modernist restaurant located in West Hollywood, Los Angeles. A 14-seat Spanish Modernist chef's coun...
  Metadata: {'source': '../data/tagged_restaurant_descriptions.txt'}

[Document 1]
  Content : "Providence is a Seafood / Contemporary restaurant located in Hollywood, Los Angeles. A celebrated seafood-forward fine ...
  Metadata: {'source': '../data/tagged_restaurant_descriptions.txt'}

[Document 2]
  Content : "Hayato is a Japanese Kaiseki restaurant located in Downtown Arts District, Los Angeles. A 7-seat intimate kaiseki resta...
  Metadata: {'source': '../data/tagged_restaurant_descriptions.txt'}


## 🤖 Cell 8 — Initialize the Embedding Model

**Model:** `sentence-transformers/all-MiniLM-L6-v2`  
- Fast, lightweight, excellent semantic understanding  
- Runs **100% locally** — no API cost during development  
- Produces 384-dimensional embeddings  
- First run will download the model (~90MB) and cache it locally

**Production alternative:** Switch to `OpenAIEmbeddings()` for higher-quality  
embeddings when deploying (see commented code below).

In [ ]:
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print(f"⏳ Loading embedding model: {EMBEDDING_MODEL}")
print("   (First run downloads ~90MB model — subsequent runs load from cache)")

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# -------------------------------------------------------
# PRODUCTION ALTERNATIVE — OpenAI text-embedding-3-small
# Higher quality, requires OPENAI_API_KEY in .env
# Cost: ~$0.00002 per 1K tokens (very cheap for 100 restaurants)
# -------------------------------------------------------
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Quick sanity check — embed a test string
test_vector = embeddings.embed_query("romantic Japanese omakase restaurant")
print(f"\n✅ Embedding model loaded successfully")
print(f"   Vector dimensions : {len(test_vector)}")
print(f"   Test vector sample: {[round(v, 4) for v in test_vector[:5]]}...")

⏳ Loading embedding model: sentence-transformers/all-MiniLM-L6-v2
   (First run downloads ~90MB model — subsequent runs load from cache)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


✅ Embedding model loaded successfully
   Vector dimensions : 384
   Test vector sample: [-0.0076, 0.0799, 0.0431, 0.0455, -0.1144]...


## 🗃️ Cell 9 — Build & Persist the ChromaDB Vector Database

Embeds all 94 restaurant documents and stores them in a local  
ChromaDB database at `./chroma_restaurants/`.  

**Run this cell once.** On subsequent runs, load from disk using Cell 10 instead.

In [9]:
CHROMA_DIR = "./chroma_restaurants"

print(f"⏳ Building vector database from {len(documents)} restaurant documents...")
print(f"   Embedding model : {EMBEDDING_MODEL}")
print(f"   Persist location: {CHROMA_DIR}")
print()

db_restaurants = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="la_restaurants"
)

collection_count = db_restaurants._collection.count()
print(f"✅ Vector database built and persisted to '{CHROMA_DIR}'")
print(f"   Documents embedded : {collection_count}")
print(f"   Collection name    : la_restaurants")

⏳ Building vector database from 94 restaurant documents...
   Embedding model : sentence-transformers/all-MiniLM-L6-v2
   Persist location: ./chroma_restaurants

✅ Vector database built and persisted to './chroma_restaurants'
   Documents embedded : 94
   Collection name    : la_restaurants


## 🔁 Cell 10 — (Optional) Reload the Database from Disk

After the database has been built (Cell 9), you can skip the embedding step  
on future runs by loading directly from the persisted ChromaDB directory.

In [10]:
# -------------------------------------------------------
# UNCOMMENT THIS CELL on subsequent notebook runs to skip
# re-embedding and load the existing database from disk.
# -------------------------------------------------------

# CHROMA_DIR = "./chroma_restaurants"
# EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

# db_restaurants = Chroma(
#     persist_directory=CHROMA_DIR,
#     embedding_function=embeddings,
#     collection_name="la_restaurants"
# )

# collection_count = db_restaurants._collection.count()
# print(f"✅ Loaded existing database from '{CHROMA_DIR}'")
# print(f"   Documents in collection: {collection_count}")

print("ℹ️  Cell 10 is commented out — using the database built in Cell 9.")
print("   Uncomment this cell on future runs to load from disk instead of re-embedding.")

ℹ️  Cell 10 is commented out — using the database built in Cell 9.
   Uncomment this cell on future runs to load from disk instead of re-embedding.


## 🔍 Cell 11 — Raw Similarity Search Test

Direct query to ChromaDB — shows the raw `Document` objects returned  
before we map them back to the full restaurant DataFrame.

In [11]:
test_query = "romantic Japanese omakase sushi restaurant"

raw_results = db_restaurants.similarity_search(test_query, k=5)

print(f"Query: '{test_query}'")
print(f"Top {len(raw_results)} raw document results:\n")
for i, doc in enumerate(raw_results, 1):
    print(f"  [{i}] {doc.page_content[:110]}...")

Query: 'romantic Japanese omakase sushi restaurant'
Top 5 raw document results:

  [1] "Sushi Kaneyoshi is a Japanese / Sushi restaurant located in Little Tokyo, Los Angeles. A highly exclusive oma...
  [2] "Shin Sushi is a Japanese Omakase restaurant located in Encino, Los Angeles. A one-Michelin-star cozy omakase ...
  [3] "Sushi Inaba is a Japanese Omakase restaurant located in Torrance, Los Angeles. A one-Michelin-star intimate o...
  [4] "Sushi Takeda is a Japanese restaurant located in Little Tokyo, Los Angeles. A Michelin Selected traditional J...
  [5] "715 Sushi is a Japanese / Sushi restaurant located in Downtown Los Angeles, Los Angeles. A minimalist omakase...


## 🗺️ Cell 12 — Name Extraction Helper

The `restaurant_metadata` text starts with the restaurant's name  
(format: `"Name is a Cuisine restaurant..."`).  
This helper parses the name from the document's page content  
to look up the full record in the DataFrame.

In [12]:
def extract_restaurant_name(page_content: str) -> str:
    """
    Extracts the restaurant name from a metadata string.
    
    The metadata format is:
        "<Name> is a <Cuisine Type> restaurant located in <Location>..."
    
    Strategy: The name is everything before " is a ".
    Falls back to matching against the full restaurants DataFrame if needed.
    """
    content = page_content.strip().strip('"')
    
    # Primary: split on ' is a ' — covers virtually all entries
    if " is a " in content:
        return content.split(" is a ")[0].strip()
    
    # Fallback: split on ' is ' for edge cases like "X is located..."
    if " is " in content:
        return content.split(" is ")[0].strip()
    
    # Last resort: return first 40 characters
    return content[:40].strip()


# Test the extraction on a few documents
print("=== NAME EXTRACTION TEST ===")
for i in [0, 3, 13, 26, 45]:
    extracted = extract_restaurant_name(documents[i].page_content)
    actual = restaurants.iloc[i]['Name']
    status = "✅" if extracted == actual else "⚠️ "
    print(f"  {status} Doc {i:>2} | Extracted: '{extracted}' | Actual: '{actual}'")

=== NAME EXTRACTION TEST ===
  ✅ Doc  0 | Extracted: 'Somni' | Actual: 'Somni'
  ✅ Doc  3 | Extracted: 'n/naka' | Actual: 'n/naka'
  ✅ Doc 13 | Extracted: 'Holbox' | Actual: 'Holbox'
  ✅ Doc 26 | Extracted: 'Maccheroni Republic' | Actual: 'Maccheroni Republic'
  ✅ Doc 45 | Extracted: 'Niku X' | Actual: 'Niku X'


## ⚙️ Cell 13 — Core Retrieval Function

`retrieve_semantic_recommendations()` is the **main search function** for this project.  
It accepts a natural language query and optional filters, queries ChromaDB,  
maps results back to the full DataFrame, and returns a ranked results table.

**Parameters:**
| Parameter | Type | Default | Description |
|-----------|------|---------|-------------|
| `query` | str | required | Natural language search query |
| `top_k` | int | 5 | Number of results to return |
| `michelin_filter` | str\|None | None | Filter: `'1-Star'`, `'2-Star'`, `'3-Star'`, `'Bib-Gourmand'`, `'Michelin-Selected'` |
| `price_filter` | str\|None | None | Filter: `'$'`, `'$$'`, `'$$$'`, `'$$$$'`, `'$$$$$'` |
| `atmosphere_filter` | str\|None | None | Filter: `'Fine-Dining'`, `'Romantic'`, `'Trendy'`, `'Casual'`, `'Smart-Casual'` |
| `rooftop_only` | bool | False | Restrict to rooftop/top-floor restaurants only |

In [13]:
def retrieve_semantic_recommendations(
    query: str,
    top_k: int = 5,
    michelin_filter: str = None,
    price_filter: str = None,
    atmosphere_filter: str = None,
    rooftop_only: bool = False,
) -> pd.DataFrame:
    """
    Retrieves the top-K semantically matched restaurants for a natural language query.

    Searches the ChromaDB vector database for the closest semantic matches,
    maps the results back to the full restaurant DataFrame, applies any
    optional post-retrieval filters, and returns a ranked results table.

    Parameters
    ----------
    query            : Natural language search (e.g. 'romantic Italian dinner')
    top_k            : Maximum number of results to return (default 5)
    michelin_filter  : Optional Michelin level filter (e.g. '1-Star', 'Bib-Gourmand')
    price_filter     : Optional price range filter (e.g. '$$$$')
    atmosphere_filter: Optional atmosphere filter (e.g. 'Romantic', 'Fine-Dining')
    rooftop_only     : If True, only return rooftop/top-floor restaurants

    Returns
    -------
    pd.DataFrame : Ranked table of matching restaurants with key columns
    """
    # Step 1: Retrieve more candidates than needed from ChromaDB
    # (pull extra so filters don't reduce results below top_k)
    candidate_pool = min(len(restaurants), max(50, top_k * 8))
    raw_docs = db_restaurants.similarity_search(query, k=candidate_pool)

    # Step 2: Extract restaurant names from document content
    matched_names = []
    seen = set()  # Deduplicate (same name can appear from multiple chunks)
    for doc in raw_docs:
        name = extract_restaurant_name(doc.page_content)
        if name and name not in seen:
            matched_names.append(name)
            seen.add(name)

    # Step 3: Look up full records from the DataFrame, preserving rank order
    results = []
    for name in matched_names:
        matches = restaurants[restaurants["Name"] == name]
        if not matches.empty:
            results.append(matches.iloc[0])

    if not results:
        print("⚠️  No results found for the given query.")
        return pd.DataFrame()

    df_results = pd.DataFrame(results).reset_index(drop=True)

    # Step 4: Apply optional post-retrieval filters
    if michelin_filter:
        df_results = df_results[df_results["Michelin-Guide"] == michelin_filter]

    if price_filter:
        df_results = df_results[df_results["Price"] == price_filter]

    if atmosphere_filter:
        # Partial match so 'Romantic' also catches 'Romantic / Smart-Casual'
        df_results = df_results[
            df_results["Dining Atmosphere"].str.contains(atmosphere_filter, case=False, na=False)
        ]

    if rooftop_only:
        df_results = df_results[df_results["Sky-High Rooftop"] == "Yes"]

    # Step 5: Return top_k results with display columns
    display_cols = [
        "Name", "Location", "Cuisine Type", "Dining Atmosphere",
        "Michelin-Guide", "Price", "Customer Ratings",
        "Sky-High Rooftop", "Reservations", "Description"
    ]

    return df_results[display_cols].head(top_k).reset_index(drop=True)


print("✅ retrieve_semantic_recommendations() defined and ready.")

✅ retrieve_semantic_recommendations() defined and ready.


## 🧪 Cell 14 — Test Query: Japanese Omakase

Testing a precise cuisine + dining style query.

In [14]:
query = "intimate Japanese omakase sushi experience"
results = retrieve_semantic_recommendations(query, top_k=5)

print(f"Query: '{query}'")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'intimate Japanese omakase sushi experience'
Results: 5 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Sushi Kaneyoshi,Little Tokyo,Japanese / Sushi,Fine-Dining,1-Star,$$$$,5.0,No,Reservation Only,A highly exclusive omakase sushi experience tucked in Little Tokyo offering pristine seasonal fi...
1,Shin Sushi,Encino,Japanese Omakase,Fine-Dining,1-Star,$$$$,4.0,No,Reservation Only,A one-Michelin-star cozy omakase den from Chef Taketoshi Azumi hidden in a strip mall offering a...
2,Sushi Takeda,Little Tokyo,Japanese,Fine-Dining,Michelin-Selected,$$$$,4.6,No,Reservation Only,A Michelin Selected traditional Japanese sushi restaurant offering an intimate omakase experienc...
3,Sushi Inaba,Torrance,Japanese Omakase,Fine-Dining,1-Star,$$$$,5.0,No,Reservation Only,A one-Michelin-star intimate omakase sushi bar from Chef Yasuhiro Hirano featuring ultra-premium...
4,715 Sushi,Downtown Los Angeles,Japanese / Sushi,Fine-Dining,1-Star,$$$$,5.0,No,Reservation Only,A minimalist omakase sushi counter in the Arts District offering a chef's choice multi-course me...


## 🧪 Cell 15 — Test Query: Romantic Dinner

Testing an occasion-based query using the atmosphere filter.

In [15]:
query = "romantic dinner with beautiful ambiance and great food"
results = retrieve_semantic_recommendations(
    query,
    top_k=5,
    atmosphere_filter="Romantic"
)

print(f"Query: '{query}' | Filter: atmosphere='Romantic'")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'romantic dinner with beautiful ambiance and great food' | Filter: atmosphere='Romantic'
Results: 5 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Night We Met,Los Angeles,Pan Asian,Romantic / Smart-Casual,No,$$$,4.4,No,No,"Located in the Wilshire LaBrea Apartments & Opened by the duo behind Met Her at a Bar, it featur..."
1,Bar di Bello,Silver Lake,Italian / Milanese,Romantic / Smart-Casual,No,$$$$,4.6,No,Yes,Bar di Bello is a Milan inspired restaurant in Silver Lake. The menu features authentic Italian ...
2,Yamashiro,Hollywood Hills,Japanese / Sushi,Romantic / Smart-Casual,No,$$$$,4.3,Yes,Yes,Yamashiro sits atop The Hollywood Hills with break-taking panoramic views of Los Angeles. The di...
3,Sora Craft Kitchen,Downtown Los Angeles,Turkish,Romantic / Smart-Casual,Michelin-Selected,$$$,4.8,No,Reservation Only,A one-man show run by Chef Okay Inak. This Turkish-born chef has a fine dining pedigree but runs...
4,Heritage,Long Beach,Californian,Romantic,1-Star,$$$,4.0,No,Reservation Only,A one-Michelin-star Californian tasting menu inside a converted Craftsman bungalow run by siblin...


## 🧪 Cell 16 — Test Query: Michelin Tasting Menu on a Budget

Testing a value-driven query with Michelin and price filters combined.

In [16]:
query = "Michelin starred tasting menu that is affordable and casual"
results = retrieve_semantic_recommendations(
    query,
    top_k=5,
    michelin_filter="Bib-Gourmand"
)

print(f"Query: '{query}' | Filter: michelin='Bib-Gourmand'")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'Michelin starred tasting menu that is affordable and casual' | Filter: michelin='Bib-Gourmand'
Results: 5 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Langer's,Westlake,Deli,Casual,Bib-Gourmand,$$,4.8,No,No,A Michelin Bib Gourmand iconic LA deli institution celebrated for its hand-cut pastrami sandwich...
1,Komal,Los Angeles,Mexican,Casual,Bib-Gourmand,$,4.8,No,Yes,A new 2025 Michelin Bib Gourmand Mexican restaurant offering deeply flavored regional Mexican di...
2,Maccheroni Republic,Downtown Los Angeles,Italian-American,Casual,Bib-Gourmand,$$,4.5,No,Yes,A Michelin Bib Gourmand Italian-American trattoria offering rustic handmade pastas and classic I...
3,The Factory Kitchen,Downtown Los Angeles,Italian,Trendy,Bib-Gourmand,$$,4.6,No,Yes,A Michelin Bib Gourmand Italian restaurant in the Arts District known for its exceptional handma...
4,Pizzeria Bianco,Downtown Los Angeles,Pizza,Casual,Bib-Gourmand,$$,4.7,No,Yes,A Michelin Bib Gourmand outpost of the legendary Phoenix pizzeria from Chris Bianco offering woo...


## 🧪 Cell 17 — Test Query: Rooftop Views

Testing the rooftop filter for users who want views with their meal.

In [17]:
query = "upscale restaurant with stunning city views and cocktails"
results = retrieve_semantic_recommendations(
    query,
    top_k=5,
    rooftop_only=True
)

print(f"Query: '{query}' | Filter: rooftop_only=True")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'upscale restaurant with stunning city views and cocktails' | Filter: rooftop_only=True
Results: 5 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Aperture at City Club LA,Downtown Los Angeles,Contemporary American,Fine-Dining,No,$$$$,4.2,Yes,Yes,A contemporary American restaurant on the 51st floor of City National Plaza offering breathtakin...
1,71Above,Downtown Los Angeles,Contemporary American,Fine-Dining,No,$$$$,4.7,Yes,Yes,A breathtaking contemporary American fine dining restaurant on the 71st floor of the US Bank Tow...
2,Yamashiro,Hollywood Hills,Japanese / Sushi,Romantic / Smart-Casual,No,$$$$,4.3,Yes,Yes,Yamashiro sits atop The Hollywood Hills with break-taking panoramic views of Los Angeles. The di...
3,Sushi Samba,West Hollywood,Japanese / Brazilian,Trendy / Smart-Casual,No,$$$,4.7,Yes,No,"A glamorous rooftop destination in West Hollywood It offers a high-energy ""clubstaurant"" vibe, v..."
4,Videre,Beverly Hills,Contemporary American,Smart-Casual,No,$$$,4.2,Yes,No,A rooftop bar in the Kimpton Hotel Wilshire that offers Costal Southern Californian cuisine meet...


## 🧪 Cell 18 — Test Query: Upscale Steakhouse

Testing a specific cuisine and price tier query.

In [18]:
query = "best steakhouse for a business dinner with premium cuts and wine"
results = retrieve_semantic_recommendations(
    query,
    top_k=5,
    price_filter="$$$$"
)

print(f"Query: '{query}' | Filter: price='$$$$'")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'best steakhouse for a business dinner with premium cuts and wine' | Filter: price='$$$$'
Results: 5 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Fleming's Steakhouse,Downtown Los Angeles,Steakhouse,Fine-Dining,No,$$$$,4.6,No,Yes,An upscale steakhouse across from Crypto.com Arena offering seven different prime steak cuts han...
1,Steak 48,Beverly Hills,Steakhouse,Fine-Dining,No,$$$$,4.6,No,Yes,"Steak 48, Founded by the Mastro brothers, is an upscale, contemporary fine-dining steakhouse kno..."
2,CUT by Wolfgang Puck,Beverly Hills,Steakhouse,Fine-Dining,No,$$$$,5.0,No,Yes,The most acclaimed steakhouse in the US from Wolfgang Puck set inside the Beverly Wilshire Hotel...
3,La Boucherie,Downtown Los Angeles,Steakhouse,Fine-Dining,No,$$$$,4.5,Yes,Yes,An elegant American steak and seafood restaurant with floor-to-ceiling panoramic LA skyline view...
4,H&H Brazilian Steakhouse,Downtown Los Angeles,Brazilian Steakhouse,Fine-Dining,No,$$$$,4.3,No,Yes,The number one Brazilian steakhouse in Los Angeles offering the highest grade Halal beef lamb an...


## 🧪 Cell 19 — Test Query: Ultra-Luxury Splurge

Testing top-tier Michelin experiences with the highest price point.

In [19]:
query = "world-class tasting menu special occasion blow out dinner Los Angeles"
results = retrieve_semantic_recommendations(
    query,
    top_k=5,
    michelin_filter="3-Star"
)

print(f"Query: '{query}' | Filter: michelin='3-Star'")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'world-class tasting menu special occasion blow out dinner Los Angeles' | Filter: michelin='3-Star'
Results: 2 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Somni,West Hollywood,Spanish Modernist,Fine-Dining,3-Star,$$$$$,5.0,No,Reservation Only,A 14-seat Spanish Modernist chef's counter offering a 20+ course tasting menu led by Chef Aitor ...
1,Providence,Hollywood,Seafood / Contemporary,Fine-Dining,3-Star,$$$$,5.0,No,Reservation Only,A celebrated seafood-forward fine dining institution from Chef Michael Cimarusti with nearly 20 ...


## 🧪 Cell 20 — Test Query: Italian Fine Dining

Testing a cuisine-specific query with no additional filters.

In [20]:
query = "Italian restaurant with exceptional pasta and elegant atmosphere"
results = retrieve_semantic_recommendations(query, top_k=6)

print(f"Query: '{query}'")
print(f"Results: {len(results)} restaurants found\n")
results

Query: 'Italian restaurant with exceptional pasta and elegant atmosphere'
Results: 6 restaurants found



,Name,Location,Cuisine Type,Dining Atmosphere,Michelin-Guide,Price,Customer Ratings,Sky-High Rooftop,Reservations,Description
0,Pasta | Bar,Encino,Italian,Fine-Dining,1-Star,$$$,3.0,No,Reservation Only,A one-Michelin-star Italian-inspired multi-course tasting menu from Chef Phillip Frankland Lee f...
1,Bad Roman,Beverly Hills,Italian / Modernist,Trendy / Smart-Casual,No,$$$,4.9,No,No,"A maximalist, highly theatrical Italian-American restaurant. Known for its energetic atmosphere,..."
2,Maccheroni Republic,Downtown Los Angeles,Italian-American,Casual,Bib-Gourmand,$$,4.5,No,Yes,A Michelin Bib Gourmand Italian-American trattoria offering rustic handmade pastas and classic I...
3,Bar di Bello,Silver Lake,Italian / Milanese,Romantic / Smart-Casual,No,$$$$,4.6,No,Yes,Bar di Bello is a Milan inspired restaurant in Silver Lake. The menu features authentic Italian ...
4,The Factory Kitchen,Downtown Los Angeles,Italian,Trendy,Bib-Gourmand,$$,4.6,No,Yes,A Michelin Bib Gourmand Italian restaurant in the Arts District known for its exceptional handma...
5,Bestia,Downtown Arts District,Italian,Trendy,No,$$$$,4.8,No,Yes,An iconic LA Italian restaurant from Chef Ori Menashe in an industrial-chic Arts District space ...


## 🧪 Cell 21 — Test Query: Similarity Search with Score

Uses `similarity_search_with_score()` to expose the raw cosine distance  
for each result — useful for tuning and debugging retrieval quality.  
**Lower score = better match** (Chroma uses L2 distance by default).

In [21]:
query = "Mexican seafood tacos casual Michelin"

scored_results = db_restaurants.similarity_search_with_score(query, k=8)

print(f"Query: '{query}'")
print(f"\n{'Rank':<5} {'Score':<10} Restaurant")
print("-" * 80)
for rank, (doc, score) in enumerate(scored_results, 1):
    name = extract_restaurant_name(doc.page_content)
    # Retrieve row for Michelin/Price info
    row = restaurants[restaurants['Name'] == name]
    michelin = row['Michelin-Guide'].values[0] if not row.empty else "?"
    price = row['Price'].values[0] if not row.empty else "?"
    print(f"{rank:<5} {score:<10.4f} {name} ({michelin}, {price})")

Query: 'Mexican seafood tacos casual Michelin'

Rank  Score      Restaurant
--------------------------------------------------------------------------------
1     0.9006     Damian (Michelin-Selected, $$$)
2     0.9201     Broken Spanish Comedor (No, $$$)
3     0.9497     The Lobster (Michelin-Selected, $$$)
4     1.0007     Pasta | Bar (1-Star, $$$)
5     1.0061     Providence (3-Star, $$$$)
6     1.0480     Holbox (1-Star, $$)
7     1.0526     Citrin (1-Star, $$$)
8     1.0747     Crustacean (Michelin-Selected, $$$$)


## 🧪 Cell 22 — Batch Query Test: Diverse Query Types

Runs a suite of natural language queries representative of what users  
will enter in the Gradio UI — validates that the retrieval function handles  
different query styles, intents, and specificity levels gracefully.

In [22]:
test_queries = [
    # Occasion-based
    "anniversary dinner with breathtaking views",
    "birthday celebration fine dining tasting menu",
    "business lunch with a professional atmosphere",
    # Cuisine-based
    "modern French cuisine with wine pairing",
    "Korean BBQ Koreatown",
    "wagyu steak premium Japanese beef",
    # Neighborhood-based  
    "Beverly Hills fine dining celebrity restaurant",
    "Downtown Los Angeles rooftop bar and dinner",
    # Experience-based
    "chef counter tasting menu sous vide innovative",
    "farm to table organic locally sourced California",
]

print("=" * 70)
print("BATCH QUERY TEST RESULTS")
print("=" * 70)

for q in test_queries:
    results = retrieve_semantic_recommendations(q, top_k=3)
    print(f"\n🔍 Query: '{q}'")
    if results.empty:
        print("   ⚠️  No results")
    else:
        for _, row in results.iterrows():
            print(f"   • {row['Name']:<35} | {row['Michelin-Guide']:<18} | {row['Price']:<6} | {row['Dining Atmosphere']}")

print("\n" + "=" * 70)
print("✅ Batch test complete")

BATCH QUERY TEST RESULTS

🔍 Query: 'anniversary dinner with breathtaking views'
   • Night We Met                        | No                 | $$$    | Romantic / Smart-Casual
   • Heritage                            | 1-Star             | $$$    | Romantic
   • 71Above                             | No                 | $$$$   | Fine-Dining

🔍 Query: 'birthday celebration fine dining tasting menu'
   • Pasta | Bar                         | 1-Star             | $$$    | Fine-Dining
   • Crustacean                          | Michelin-Selected  | $$$$   | Fine-Dining
   • Bad Roman                           | No                 | $$$    | Trendy / Smart-Casual

🔍 Query: 'business lunch with a professional atmosphere'
   • Orsa & Winston                      | 1-Star             | $$$$   | Fine-Dining
   • Spago by Wolfgang Puck              | No                 | $$$$   | Fine-Dining
   • Meteora                             | 1-Star             | $$$    | Trendy

🔍 Query: 'modern French 

## 📊 Cell 23 — Vector Database Stats & Collection Info

In [23]:
print("=" * 55)
print("   VECTOR DATABASE SUMMARY")
print("=" * 55)
print(f"  Collection name    : la_restaurants")
print(f"  Persist directory  : {CHROMA_DIR}")
print(f"  Documents embedded : {db_restaurants._collection.count()}")
print(f"  Embedding model    : {EMBEDDING_MODEL}")
print(f"  Vector dimensions  : 384")
print()
print("  Michelin coverage:")
for m in ['3-Star', '2-Star', '1-Star', 'Bib-Gourmand', 'Michelin-Selected', 'No']:
    count = (restaurants['Michelin-Guide'] == m).sum()
    print(f"    {m:<22}: {count} restaurants")
print()
print(f"  Total restaurants  : {len(restaurants)}")
print(f"  Unique neighborhoods: {restaurants['Location'].nunique()}")
print(f"  Unique cuisine types: {restaurants['Cuisine Type'].nunique()}")
print("=" * 55)
print("\n🚀 Vector database ready!")
print("   Next step: Notebook 3 — Cuisine Classification & Sentiment Analysis")
print("   After that: Notebook 4 — Gradio Dashboard UI")

   VECTOR DATABASE SUMMARY
  Collection name    : la_restaurants
  Persist directory  : ./chroma_restaurants
  Documents embedded : 94
  Embedding model    : sentence-transformers/all-MiniLM-L6-v2
  Vector dimensions  : 384

  Michelin coverage:
    3-Star                : 2 restaurants
    2-Star                : 4 restaurants
    1-Star                : 21 restaurants
    Bib-Gourmand          : 10 restaurants
    Michelin-Selected     : 15 restaurants
    No                    : 42 restaurants

  Total restaurants  : 94
  Unique neighborhoods: 30
  Unique cuisine types: 59

🚀 Vector database ready!
   Next step: Notebook 3 — Cuisine Classification & Sentiment Analysis
   After that: Notebook 4 — Gradio Dashboard UI
